# *Task 1 — Audit Kualitas Data Awal*

In [ ]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print("Shape:", df.shape)
print()
print("Tipe data:")
print(df.dtypes)
print()
print("5 baris pertama:")
df.head()

Shape: (7043, 21)

Tipe data:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

5 baris pertama:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Cek missing values standar (NaN)
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Kolom dengan missing values (NaN):")
print(missing)

Kolom dengan missing values (NaN):
Series([], dtype: int64)


In [ ]:
# Cek TotalCharges — seharusnya numerik
# Cari baris yang tidak bisa dikonversi ke float
problematic = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]
print(f"Baris bermasalah di TotalCharges: {len(problematic)}")
print(problematic[['customerID', 'TotalCharges', 'tenure']].head(20))

Baris bermasalah di TotalCharges: 11
      customerID TotalCharges  tenure
488   4472-LVYGI                    0
753   3115-CZMZD                    0
936   5709-LVOEQ                    0
1082  4367-NUYAO                    0
1340  1371-DWPAZ                    0
3331  7644-OMVMY                    0
3826  3213-VVOLG                    0
4380  2520-SGTTA                    0
5218  2923-ARZLG                    0
6670  4075-WKNIU                    0
6754  2775-SEFEE                    0


In [ ]:
# Cek SeniorCitizen — apakah ada nilai selain 0 dan 1
nilai_lain = df[~df['SeniorCitizen'].isin([0, 1])]
print(f"Nilai di luar 0/1 di SeniorCitizen: {len(nilai_lain)}")
print(df['SeniorCitizen'].unique())

Nilai di luar 0/1 di SeniorCitizen: 0
[0 1]


*Ringkasan Audit Kualitas Data:*

| Kolom | Jenis Masalah | Baris Terdampak |
|---|---|---|
| `TotalCharges` | Tipe object padahal seharusnya numerik; berisi string spasi/kosong | 11 baris |
| `SeniorCitizen` | Numerik (0/1) tapi tidak ada masalah nilai di luar range | 0 baris |

Tidak ada NaN standar di dataset ini. Masalah utama ada di `TotalCharges` yang menyimpan nilai spasi sebagai string.

# *Task 2 — Deteksi Non-Standard dan Unexpected Missing Values*

In [ ]:
# Cek unique values tiap kolom object
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col}: {df[col].unique()}")


customerID: <StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str

gender: <StringArray>
['Female', 'Male']
Length: 2, dtype: str

Partner: <StringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents: <StringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService: <StringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines: <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

OnlineSecurity: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

OnlineBackup: <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

DeviceProtection: <StringArray>
['No', 'Yes', 'No internet 

D:\Users\bsi80274\AppData\Local\Temp\ipykernel_31940\494445720.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


*Nilai placeholder yang ditemukan:*
- `TotalCharges`: berisi string spasi `' '` yang seharusnya NaN
- Beberapa kolom layanan (seperti `OnlineSecurity`, `OnlineBackup`, dll.) punya nilai `'No internet service'` dan `'No phone service'`

*Apakah 'No internet service' / 'No phone service' adalah missing value?*

Tidak, nilai-nilai ini adalah *kategori valid* Artinya pelanggan memang tidak berlangganan layanan internet atau telepon, bukan data yang hilang. Jadi tidak perlu diperlakukan sebagai missing value.

In [ ]:
# Cek tenure = 0 (pelanggan baru yang baru join, mungkin tidak masuk akal)
tenure_nol = df[df['tenure'] == 0]
print(f"Jumlah baris dengan tenure = 0: {len(tenure_nol)}")
print(tenure_nol[['customerID', 'tenure', 'TotalCharges', 'MonthlyCharges']].head())

Jumlah baris dengan tenure = 0: 11
      customerID  tenure TotalCharges  MonthlyCharges
488   4472-LVYGI       0                        52.55
753   3115-CZMZD       0                        20.25
936   5709-LVOEQ       0                        80.85
1082  4367-NUYAO       0                        25.75
1340  1371-DWPAZ       0                        56.05


# *Task 3 — Penanganan Missing Values*

In [ ]:
# Konversi TotalCharges ke float, string kosong/spasi jadi NaN otomatis
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Tipe TotalCharges sekarang:", df['TotalCharges'].dtype)
print("Jumlah NaN di TotalCharges:", df['TotalCharges'].isnull().sum())

Tipe TotalCharges sekarang: float64
Jumlah NaN di TotalCharges: 11


In [ ]:
# Isi NaN TotalCharges pakai median
median_tc = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
print("Median TotalCharges:", median_tc)

Median TotalCharges: 1397.475


*Kenapa pakai median, bukan mean?*

Kolom `TotalCharges` kemungkinan punya distribusi yang skewed (miring ke kanan) karena ada pelanggan lama dengan tagihan total sangat besar. Mean akan ikut tertarik ke nilai ekstrem tersebut, sehingga tidak merepresentasikan nilai "tengah" yang sebenarnya. Median lebih robust terhadap outlier dan lebih aman dipakai untuk imputasi data seperti ini.

In [ ]:
# Ganti tenure = 0 jadi NaN, lalu isi dengan median groupby InternetService & PhoneService
df['tenure'] = df['tenure'].replace(0, np.nan)

median_tenure = df.groupby(['InternetService', 'PhoneService'])['tenure'].transform('median')
df['tenure'] = df['tenure'].fillna(median_tenure)

print("Tenure 0 sudah diganti dengan median per grup layanan.")

Tenure 0 sudah diganti dengan median per grup layanan.


In [ ]:
# Verifikasi — pastikan tidak ada missing values di kedua kolom
print("Missing values setelah cleaning:")
print("TotalCharges:", df['TotalCharges'].isnull().sum())
print("tenure:", df['tenure'].isnull().sum())

Missing values setelah cleaning:
TotalCharges: 0
tenure: 0


# *Task 4 — Laporan Cleaning*

*Ringkasan Data Cleaning:*

```
Kolom          | Masalah Ditemukan              | Penanganan                    | Baris Terdampak
TotalCharges   | String kosong/spasi            | Konversi + isi median         | 11 baris
tenure         | Nilai 0 tidak valid            | Ganti NaN + isi groupby median| 11 baris
```

Dataset awal tidak memiliki NaN standar. Masalah utama berupa string spasi di `TotalCharges` yang menyebabkan kolom tidak bisa langsung digunakan sebagai numerik. Setelah proses cleaning, kedua kolom sudah bersih dan siap digunakan untuk modeling.